# **Análise do Uber Data Analytics**

## **Fase 1: Visão Estrutural**

Nesta fase, o objetivo é carregar os dados e realizar uma inspeção técnica para entender sua estrutura, formato e tipos de dados.

### **1.1. Configuração do Ambiente:**

Importa e inicia a SparkSession, que é a porta de entrada principal para trabalhar com DataFrames no PySpark.

- Na criação da sessão:
  - `SparkSession.builder`: é o construtor para criar uma nova sessão Spark;
  - `.appName("UberCSV")`: define o nome da aplicação Spark;
  - `.getOrCreate()`: cria a sessão Spark se não existir nenhuma ativa.

In [2]:
from pyspark.sql import SparkSession

# Criar uma SparkSession
spark = SparkSession.builder \
    .appName("UberCSV") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/18 17:43:34 WARN Utils: Your hostname, deskbob, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/10/18 17:43:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/18 17:43:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **1.2. Carregamento dos Dados:**

- Carregar o arquivo CSV em um DataFrame, onde:
  - `spark.read.csv()`: é o método do PySpark para ler arquivos CSV e criar um DataFrame;
  - `uber-dataset.csv`: o caminho do arquivo CSV para ler. Pode ser relativo ou absoluto;
  - `header=True`: indica que a primeira linha contém os nomes das colunas;
  - `inferSchema=True`: faz o Spark tentar identificar automaticamente o tipo de cada coluna (inteiro, string, etc.);
  - `sep=","`: define o separador, que por padrão é ",".

In [3]:
df = spark.read.csv(
    "uber-dataset.csv",
    header=True,
    inferSchema=True,
    sep=","
)

### **1.3. Inspeção Inicial:**

- Para obter uma amostra do conteúdo:
  - `.head(5)`: retorna as 5 primeiras linhas;
  - `.tail(5)`: retorna as 5 últimas linhas;
  - `spark.createDataFrame()`: recria um mini DataFrame só com essas linhas;
  - `.show()`: serve para exibir o conteúdo de um DataFrame Spark de forma tabular no console.

In [4]:
print("===== Primeiras 5 linhas do Dataset =====")
spark.createDataFrame(df.head(5)).show()
print("===== Últimas 5 linhas do Dataset =====")
spark.createDataFrame(df.tail(5)).show()

===== Primeiras 5 linhas do Dataset =====


+----------+-------------------+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+
|      Date|               Time|      Booking ID| Booking Status|     Customer ID| Vehicle Type|    Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|
+----------+-------------------+----------------+---------------+----------------+-------------+-------------------+-----------------+--------+--------+---------------------------+---------------------------------+--------------------

+----------+-------------------+----------------+--------------+----------------+-------------+--------------------+-----------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+
|      Date|               Time|      Booking ID|Booking Status|     Customer ID| Vehicle Type|     Pickup Location|    Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|
+----------+-------------------+----------------+--------------+----------------+-------------+--------------------+-----------------+--------+--------+---------------------------+---------------------------------+--------------------

- No PySpark, não existe o atributo `.shape como no pandas`, então para verificar a dimensão do dataset:
  - `.count()`: retorna um inteiro exato da quantidade de linhas do DataFrame;
  - `.columns`: retorna uma lista com os nomes das colunas do DataFrame;
  - `len(df.columns)`: dá o número total de colunas;
  - `{num_linhas:,}`: adiciona separador de milhares;
  - `.replace(",", ".")`: troca as vírgulas por pontos, deixando no padrão brasileiro.

In [5]:
num_linhas = df.count()
num_colunas = len(df.columns)
print(f"O Dataset possui {num_linhas:,} linhas e {num_colunas} colunas.".replace(",", "."))

O Dataset possui 150.000 linhas e 21 colunas.


- Para listar os nomes de todas as colunas:
  - `(i+1, c) for i, c in enumerate(df.columns)`: cria uma lista de tuplas, cada tupla com o nome de uma coluna e com números de índice para cada coluna;
  - `spark.createDataFrame()`: recria um mini DataFrame só com essas linhas;
  - `.show()`: serve para exibir o conteúdo de um DataFrame Spark de forma tabular no console;
  - `n=df.count()`: usado para tirar o limite de exibição de 20 linhas e exibir todas as colunas;
  - `truncate=False`: mostra tudo sem cortar nomes longos.

In [6]:
mini_df_index = spark.createDataFrame([(i+1, c) for i, c in enumerate(df.columns)], ["#","Colunas"])
mini_df_index.show(n=df.count(), truncate=False)

+---+---------------------------------+
|#  |Colunas                          |
+---+---------------------------------+
|1  |Date                             |
|2  |Time                             |
|3  |Booking ID                       |
|4  |Booking Status                   |
|5  |Customer ID                      |
|6  |Vehicle Type                     |
|7  |Pickup Location                  |
|8  |Drop Location                    |
|9  |Avg VTAT                         |
|10 |Avg CTAT                         |
|11 |Cancelled Rides by Customer      |
|12 |Reason for cancelling by Customer|
|13 |Cancelled Rides by Driver        |
|14 |Driver Cancellation Reason       |
|15 |Incomplete Rides                 |
|16 |Incomplete Rides Reason          |
|17 |Booking Value                    |
|18 |Ride Distance                    |
|19 |Driver Ratings                   |
|20 |Customer Rating                  |
|21 |Payment Method                   |
+---+---------------------------------+


- Para verificar se existem nomes duplicados:
  - `Counter()`: é como um dicionário especializado que conta quantas vezes cada elemento aparece em uma lista;
  - `.columns`: retorna uma lista com os nomes das colunas do DataFrame;
  - `col for col, count in contagem.items() if count > 1`: essa linha cria uma lista com apenas os nomes duplicados;
  - `if duplicadas`: em Python, listas vazias são avaliadas como `False`.

In [7]:
from collections import Counter

contagem = Counter(df.columns)

duplicadas = [col for col, count in contagem.items() if count > 1]

if duplicadas:
    print("Existem nomes duplicados!\nColunas duplicadas:", duplicadas)
else:
    print("Todos os nomes são únicos.")

Todos os nomes são únicos.


- Para verificar se existem nomes com espaços desnecessários:
  - `col.strip()`: remove espaços no início e no final da string;
  - `col != col.strip()`: retorna `True` se houver algum espaço extra;
  - `[col for col in df.columns if col != col.strip()]`: cria uma lista com todas as colunas problemáticas;
  - `if colunas_com_espacos`: em Python, listas vazias são avaliadas como `False`.

In [8]:
# Lista de colunas com espaços no início ou no fim
colunas_com_espacos = [col for col in df.columns if col != col.strip()]

if colunas_com_espacos:
    print("Existem colunas com espaços desnecessários:", colunas_com_espacos)
else:
    print("Nenhuma coluna possui espaços desnecessários.")

Nenhuma coluna possui espaços desnecessários.


### **1.4. Análise de Tipos de Dados:**

- No PySpark, não existe `.info()` nativamente. Então para obter os tipos de colunas e esquema:
  - `.schema()`: é o esquema do DataFrame (nomes e tipos das colunas);
  - `.schema.fields`: obtém nomes e tipos das colunas do esquema.

In [9]:
print(f"{'Coluna':<33} | {'Tipo':<15} | {'Anulável':<10}")
print("-" * (33 + 1) + "+" + "-" * (15 + 2) + "+" + "-" * (10))
for field in df.schema.fields:
    nome = field.name
    tipo = str(field.dataType)
    nullable = str(field.nullable)
    print(f"{nome:<33} | {tipo:<15} | {nullable:<10}")

Coluna                            | Tipo            | Anulável  
----------------------------------+-----------------+----------
Date                              | DateType()      | True      
Time                              | TimestampType() | True      
Booking ID                        | StringType()    | True      
Booking Status                    | StringType()    | True      
Customer ID                       | StringType()    | True      
Vehicle Type                      | StringType()    | True      
Pickup Location                   | StringType()    | True      
Drop Location                     | StringType()    | True      
Avg VTAT                          | StringType()    | True      
Avg CTAT                          | StringType()    | True      
Cancelled Rides by Customer       | StringType()    | True      
Reason for cancelling by Customer | StringType()    | True      
Cancelled Rides by Driver         | StringType()    | True      
Driver Cancellation Reason

- E para obter a contagem de valores não-nulos:
  - `for c in df.columns`: Percorre todas as colunas do DataFrame `df.columns`;
  - `col(c)`: cria uma referência à coluna `c` no PySpark;
  - `isNotNull()`: retorna True apenas para valores que não são nulos (`None` ou `null`);
  - `trim()`: remove espaços no início e no fim da string garantindo que strings vazias ou apenas espaços não sejam contadas como válidas;
  - `lower()`: converte a string para minúsculas;
  - `when(condição, c)`: se a condição for `True` mantém o valor da coluna, senão será tratado como nulo;
  - `count()`: conta o número de valores não nulos resultantes;
  - `.alias(c)`: mantêm o nome original da coluna;
  - `.select()`: cria um novo DataFrame com uma coluna para cada coluna original, mostrando quantos valores válidos existem;
  - `.show()`: exibe o resultado.

In [41]:
from pyspark.sql.functions import count, col, trim, lower, when

df.select([
    count(
        when(
            (col(c).isNotNull()) & 
            (trim(col(c)) != "") & 
            (lower(trim(col(c))) != "nan"),
            c
        )
    ).alias(c)
    for c in df.columns
]).show()

+------+------+----------+--------------+-----------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+--------------+---------------+--------------+
|  Date|  Time|Booking ID|Booking Status|Customer ID|Vehicle Type|Pickup Location|Drop Location|Avg VTAT|Avg CTAT|Cancelled Rides by Customer|Reason for cancelling by Customer|Cancelled Rides by Driver|Driver Cancellation Reason|Incomplete Rides|Incomplete Rides Reason|Booking Value|Ride Distance|Driver Ratings|Customer Rating|Payment Method|
+------+------+----------+--------------+-----------+------------+---------------+-------------+--------+--------+---------------------------+---------------------------------+-------------------------+--------------------------+----------------+-----------------------+-------------+-------------+------------

- E para verificar se os tipos de dados estão corretos, será comparado a descrição da coluna com os `dtypes` no dataset.
- Segue a tabela com os nomes das colunas e as suas respectivas descrições:

| Column Name | Description |
|:------------|:------------|
| Date | Date of the booking |
| Time | Time of the booking |
| Booking ID | Unique identifier for each ride booking |
| Booking Status | Status of booking (Completed, Cancelled by Customer, Cancelled by Driver, etc.) |
| Customer ID | Unique identifier for customers |
| Vehicle Type | Type of vehicle (Go Mini, Go Sedan, Auto, eBike/Bike, UberXL, Premier Sedan) |
| Pickup Location | Starting location of the ride |
| Drop Location | Destination location of the ride |
| Avg VTAT | Average time for driver to reach pickup location (in minutes) |
| Avg CTAT | Average trip duration from pickup to destination (in minutes) |
| Cancelled Rides by Customer | Customer-initiated cancellation flag |
| Reason for cancelling by Customer | Reason for customer cancellation |
| Cancelled Rides by Driver | Driver-initiated cancellation flag |
| Driver Cancellation Reason | Reason for driver cancellation |
| Incomplete Rides | Incomplete ride flag |
| Incomplete Rides Reason | Reason for incomplete rides |
| Booking Value | Total fare amount for the ride |
| Ride Distance | Distance covered during the ride (in km) |
| Driver Ratings |Rating given to driver (1-5 scale) |
| Customer Rating | Rating given by customer (1-5 scale) |
| Payment Method | Method used for payment (UPI, Cash, Credit Card, Uber Wallet, Debit Card) |

- Na verificação:
  - `tipos_esperados`: contém o tipo de dado ideal esperado para cada coluna baseado na descrição do dataset;
  - `if tipo_esperado:`: garante que só verifique colunas conhecidas.

In [21]:
# Tipos esperados
tipos_esperados = {
    "Date": "date",
    "Time": "timestamp",
    "Booking ID": "string",
    "Booking Status": "string",
    "Customer ID": "string",
    "Vehicle Type": "string",
    "Pickup Location": "string",
    "Drop Location": "string",
    "Avg VTAT": "float",
    "Avg CTAT": "float",
    "Cancelled Rides by Customer": "boolean",
    "Reason for cancelling by Customer": "string",
    "Cancelled Rides by Driver": "boolean",
    "Driver Cancellation Reason": "string",
    "Incomplete Rides": "boolean",
    "Incomplete Rides Reason": "string",
    "Booking Value": "double",
    "Ride Distance": "float",
    "Driver Ratings": "float",
    "Customer Rating": "float",
    "Payment Method": "string"
}

# Comparar tipos atuais x esperados
for nome, tipo_atual in df.dtypes:
    tipo_esperado = tipos_esperados.get(nome)
    if tipo_esperado:
        if tipo_atual == tipo_esperado:
            print(f"{nome:<35} OK  ({tipo_atual})")
        else:
            print(f"{nome:<35} X   ({tipo_atual}) → esperado ({tipo_esperado})")


Date                                OK  (date)
Time                                OK  (timestamp)
Booking ID                          OK  (string)
Booking Status                      OK  (string)
Customer ID                         OK  (string)
Vehicle Type                        OK  (string)
Pickup Location                     OK  (string)
Drop Location                       OK  (string)
Avg VTAT                            X   (string) → esperado (float)
Avg CTAT                            X   (string) → esperado (float)
Cancelled Rides by Customer         X   (string) → esperado (boolean)
Reason for cancelling by Customer   OK  (string)
Cancelled Rides by Driver           X   (string) → esperado (boolean)
Driver Cancellation Reason          OK  (string)
Incomplete Rides                    X   (string) → esperado (boolean)
Incomplete Rides Reason             OK  (string)
Booking Value                       X   (string) → esperado (double)
Ride Distance                       X   (stri

## **Fase 2: Visão de Qualidade (Diagnóstico e Limpeza dos Dados)**

O foco aqui é identificar problemas de qualidade e consistência nos dados para garantir que a análise subsequente seja precisa.

### **2.1. Detecção de Valores Ausentes:**